In [1]:
import gbd_mapping
import risk_distributions
import pathlib
import pandas as pd, numpy as np
import vivarium_inputs
from vivarium_inputs import utility_data, globals as vi_globals, utilities as vi_utils
from vivarium_gbd_access import gbd
import os, contextlib, warnings, loguru
from lsff_utils.hemoglobin_distribution import hemoglobin_cdf_from_mean_sd

from vivarium_inputs.validation.raw import DataDoesNotExistError, DataAbnormalError
from tqdm.notebook import tqdm

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


# Non-pregnant anemia

In [2]:
pd.set_option("display.max_columns", 30)

In [3]:
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

In [4]:
location = "india"
vehicle = "rice"
intervention_scenario = "intervention"

In [5]:
# Parameters
location = "nigeria"
vehicle = "bouillon"
intervention_scenario = "intervention"


## Setup and scenarios

In [6]:
index_cols = ["sex", "age_start", "age_end", "wealth_quintile"]

age_group_ids = [
    2,3,
    388,389,
    6,7,8,9,10,11,12,13,14,15,16,17,18,19,20, 30, 31, 32, 235
]
sex_ids = [1,2]

DRAWS = [f'draw_{i}' for i in range(500)] # NOTE: Some GBD 2021 things return 1,000 but others don't

In [7]:
effective_baseline_coverage = (
    pd.read_csv(f'../0100_data_prep/results/iron/{vehicle}/baseline_fortification/effective_coverage/{location}.csv')
)
assert (effective_baseline_coverage.vehicle_name == vehicle).all()
effective_baseline_coverage = effective_baseline_coverage.drop(columns=["vehicle_name"])
effective_baseline_coverage

,wealth_quintile,value
0,fourth,0.535497
1,highest,0.559838
2,lowest,0.415369
3,middle,0.510484
4,second,0.467278


In [8]:
def expand(df):
    for col in sorted(list(set(df.columns) - {'value'})):
        if df[col].isnull().any():
            df = pd.concat([
                df[df[col].notnull()],
                *[df[df[col].isnull()].assign(**{col: value}) for value in df[df[col].notnull()][col].unique()]
            ])
    
    return df

In [9]:
for col, fill_value in [("age_start", 0), ("age_end", 125)]:
    if col not in effective_baseline_coverage.columns:
        effective_baseline_coverage[col] = fill_value
    else:
        effective_baseline_coverage[col] = effective_baseline_coverage[col].fillna(fill_value)

In [10]:
effective_baseline_coverage = expand(effective_baseline_coverage)
effective_baseline_coverage

,wealth_quintile,value,age_start,age_end
0,fourth,0.535497,0,125
1,highest,0.559838,0,125
2,lowest,0.415369,0,125
3,middle,0.510484,0,125
4,second,0.467278,0,125


In [11]:
effective_counterfactual_coverage = (
    pd.read_csv(f'../0100_data_prep/results/iron/{vehicle}/{intervention_scenario}/intervention_fortification/effective_coverage/{location}.csv')
)
assert (effective_counterfactual_coverage.vehicle_name == vehicle).all()
effective_counterfactual_coverage = effective_counterfactual_coverage.drop(columns=["vehicle_name"])
effective_counterfactual_coverage

,wealth_quintile,sex,value
0,lowest,Female,0.618069
1,second,Female,0.626300
2,middle,Female,0.627742
3,fourth,Female,0.632860
4,highest,Female,0.629615
5,lowest,Male,0.618069
6,second,Male,0.626300
7,middle,Male,0.627742
8,fourth,Male,0.632860
9,highest,Male,0.629615


In [12]:
for col, fill_value in [("age_start", 0), ("age_end", 125)]:
    if col not in effective_counterfactual_coverage.columns:
        effective_counterfactual_coverage[col] = fill_value
    else:
        effective_counterfactual_coverage[col] = effective_counterfactual_coverage[col].fillna(fill_value)

In [13]:
effective_counterfactual_coverage = expand(effective_counterfactual_coverage)
effective_counterfactual_coverage

,wealth_quintile,sex,value,age_start,age_end
0,lowest,Female,0.618069,0,125
1,second,Female,0.626300,0,125
2,middle,Female,0.627742,0,125
3,fourth,Female,0.632860,0,125
4,highest,Female,0.629615,0,125
5,lowest,Male,0.618069,0,125
6,second,Male,0.626300,0,125
7,middle,Male,0.627742,0,125
8,fourth,Male,0.632860,0,125
9,highest,Male,0.629615,0,125


In [14]:
population = (
    pd.read_csv(f'../0100_data_prep/results/population/stratified/{location}.csv')
)

In [15]:
non_pregnant_pop = population.pipe(lambda df: df[df.pregnant == "not_pregnant"]).drop(columns="pregnant")
non_pregnant_pop = non_pregnant_pop.set_index([c for c in non_pregnant_pop.columns if c != 'value']).value
non_pregnant_pop

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    lowest             16960.496219
                               second             16890.884668
                               middle             15946.960379
                               fourth             14017.956071
                               highest            13029.810174
                                                      ...     
Male    95.0       125.000000  lowest              1846.968043
                               second              1616.329970
                               middle              1661.447043
                               fourth              1779.062429
                               highest             1896.973078
Name: value, Length: 250, dtype: float64

In [16]:
population_age_groups = non_pregnant_pop.reset_index()[["age_start", "age_end"]].drop_duplicates().sort_values("age_start")
population_age_groups

,age_start,age_end
0,0.000000,0.019178
5,0.019178,0.076712
10,0.076712,0.500000
15,0.500000,1.000000
20,1.000000,2.000000
25,2.000000,5.000000
30,5.000000,10.000000
35,10.000000,15.000000
40,15.000000,20.000000
45,20.000000,25.000000


In [17]:
def map_to_population_age_groups(df):
    result = (
        population_age_groups.merge(df, how="cross", suffixes=("", "_orig"))
            .pipe(lambda df: df[(df.age_end <= df.age_end_orig) & (df.age_start >= df.age_start_orig)])
            .drop(columns=["age_start_orig", "age_end_orig"])
    )
    return result

In [18]:
effective_baseline_coverage = map_to_population_age_groups(effective_baseline_coverage).set_index([c for c in effective_baseline_coverage.columns if c != 'value']).value
effective_counterfactual_coverage = map_to_population_age_groups(effective_counterfactual_coverage).set_index([c for c in effective_counterfactual_coverage.columns if c != 'value']).value

In [19]:
delta_effective_coverage = effective_counterfactual_coverage.sub(effective_baseline_coverage)
delta_effective_coverage

wealth_quintile  age_start  age_end     sex   
fourth           0.000000   0.019178    Female    0.097363
                                        Male      0.097363
                 0.019178   0.076712    Female    0.097363
                                        Male      0.097363
                 0.076712   0.500000    Female    0.097363
                                                    ...   
second           85.000000  90.000000   Male      0.159021
                 90.000000  95.000000   Female    0.159021
                                        Male      0.159021
                 95.000000  125.000000  Female    0.159021
                                        Male      0.159021
Name: value, Length: 250, dtype: float64

In [20]:
def reshape_to_vivarium_format(df, location):
    df = vi_utils.reshape(df, value_cols=[c for c in df.columns if 'draw_' in c])
    df = vi_utils.scrub_gbd_conventions(df, location)
    df = vi_utils.split_interval(df, interval_column="age", split_column_prefix="age")
    df = vi_utils.split_interval(df, interval_column="year", split_column_prefix="year")
    df = vi_utils.sort_hierarchical_data(df)
    df.index = df.index.droplevel("location")
    return df

## Pull GBD hemoglobin distributions

In [21]:
me_ids = {
    "hemoglobin_mean": 10487,
    "hemoglobin_sd": 10488,
}

In [22]:
def get_modelable_entity_draws(me_id, location):
    location_id = utility_data.get_location_id(location.title())
    result = gbd.get_modelable_entity_draws(me_id=me_id, location_id=location_id, year_id=2021)
    return (
        reshape_to_vivarium_format(result, location.title())
            .droplevel(["year_start", "year_end", "measure_id", "metric_id", "model_version_id", "modelable_entity_id"])[DRAWS]
            .copy()
    )

In [23]:
hgb_mean = get_modelable_entity_draws(me_ids["hemoglobin_mean"], location)
hgb_mean

draw_0      draw_1      draw_2      draw_3  \
sex    age_start age_end                                                      
Female 0.000000  0.019178    140.203331  139.700680  137.486100  143.738463   
       0.019178  0.076712    121.992596  122.837138  123.877519  120.604868   
       0.076712  0.500000    104.152465  104.343166  100.933682  103.672346   
       0.500000  1.000000    100.148802   99.844921   99.645127  100.145850   
       1.000000  2.000000    100.955723  100.339208  102.148597  101.234802   
       2.000000  5.000000    105.233662  104.333100  102.835176  105.304765   
       5.000000  10.000000   119.455834  112.957619  115.682929  114.256919   
       10.000000 15.000000   119.312919  106.273992  111.855854  122.097443   
       15.000000 20.000000   114.861226  117.794304  116.359602  117.648438   
       20.000000 25.000000   116.232257  117.251310  116.393998  114.894111   
       25.000000 30.000000   116.082332  114.626063  116.408492  116.811736   
       30.000000 35.000000   115.103530  115.163515  116.767807  116.269128   
       35.000000 40.000000   115.986054  113.528076  114.852973  114.288218   
       40.000000 45.000000   115.257500  117.940598  117.278455  116.853328   
       45.000000 50.000000   118.568757  119.048450  117.278407  117.835148   
       50.000000 55.000000   115.669641  121.096756  111.909301  123.510802   
       55.000000 60.000000   116.888277  111.934668  117.516013  116.335911   
       60.000000 65.000000   117.073529  120.448205  112.138824  119.361783   
       65.000000 70.000000   120.936133  131.305616  125.299052  121.720141   
       70.000000 75.000000   120.728706  120.069195  124.432953  117.908148   
       75.000000 80.000000   126.020295  126.733554  121.556975  127.985198   
       80.000000 85.000000   126.158836  126.248607  116.366249  123.473284   
       85.000000 90.000000   118.780034  118.418548  114.061451  115.310723   
       90.000000 95.000000   109.477010  104.789317  110.610979  108.944244   
       95.000000 125.000000  102.971577  100.970692   99.657313   98.358658   
Male   0.000000  0.019178    139.655000  152.652888  141.405358  135.236759   
       0.019178  0.076712    111.875330  122.227176  119.526917  118.369764   
       0.076712  0.500000     94.267052   98.335982   96.967303   98.720429   
       0.500000  1.000000    100.092403   98.678200   93.913129   99.908232   
       1.000000  2.000000    100.226883   94.470196   98.331789   95.255325   
       2.000000  5.000000    101.069093  107.942797  103.555610  102.374253   
       5.000000  10.000000   119.838610  110.970261  109.073412  112.143982   
       10.000000 15.000000   129.595152  119.217013  127.657332  129.642930   
       15.000000 20.000000   129.129905  144.220990  141.441577  142.995724   
       20.000000 25.000000   151.050744  131.461208  146.612140  132.307327   
       25.000000 30.000000   132.425569  149.591271  142.560174  135.773968   
       30.000000 35.000000   140.605288  141.846950  135.968545  160.662459   
       35.000000 40.000000   138.202586  145.354328  144.617087  133.079794   
       40.000000 45.000000   141.806468  145.535717  136.109962  148.130476   
       45.000000 50.000000   133.416092  136.101393  150.337119  141.486861   
       50.000000 55.000000   144.695059  135.210554  139.837715  139.728713   
       55.000000 60.000000   138.257682  142.365263  142.266348  143.671582   
       60.000000 65.000000   137.633530  133.990371  143.313102  140.484227   
       65.000000 70.000000   143.947810  146.078428  140.887869  134.047618   
       70.000000 75.000000   141.462864  144.115084  130.980064  145.225366   
       75.000000 80.000000   125.293565  127.635808  121.016674  127.115747   
       80.000000 85.000000   136.438006  140.853010  131.367813  139.943361   
       85.000000 90.000000   133.029351  120.328832  136.611180  126.391697   
       90.000000 95.000000   116.866389  104.222078  111.737153  113.207507   
    

In [24]:
hemoglobin_mean_disparities = pd.read_csv(f'../0100_data_prep/results/hemoglobin/mean_disparities/{location}.csv')
hemoglobin_mean_disparities = (
    map_to_population_age_groups(hemoglobin_mean_disparities[hemoglobin_mean_disparities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))
        .set_index(["sex", "age_start", "age_end", "wealth_quintile"]).value
)
hemoglobin_mean_disparities

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    lowest              97.093264
                               second              99.647180
                               middle             102.658490
                               fourth             103.792784
                               highest            107.778540
                                                     ...    
Male    95.0       125.000000  lowest             113.444966
                               second             115.420130
                               middle             116.087185
                               fourth             116.578539
                               highest            118.793246
Name: value, Length: 250, dtype: float64

In [25]:
wealth_quintile_probabilities = pd.read_csv(f'../0100_data_prep/results/wealth_quintile_probabilities/{location}.csv')
wealth_quintile_probabilities

,sex,age_start,age_end,pregnant,lowest,second,middle,fourth,highest
0,Female,0.0,5.0,not_pregnant,0.220707,0.219801,0.207518,0.182416,0.169557
1,Female,5.0,15.0,not_pregnant,0.222903,0.211314,0.200941,0.189893,0.174949
2,Female,15.0,30.0,not_pregnant,0.165248,0.194708,0.200403,0.223650,0.215992
3,Female,15.0,30.0,pregnant,0.229568,0.269087,0.218290,0.169175,0.113880
4,Female,30.0,50.0,not_pregnant,0.167414,0.176150,0.188366,0.214621,0.253449
5,Female,30.0,50.0,pregnant,0.247900,0.197054,0.182014,0.163763,0.209268
6,Female,50.0,125.0,not_pregnant,0.196077,0.184902,0.228885,0.192974,0.197161
7,Male,0.0,5.0,not_pregnant,0.214963,0.223211,0.203137,0.188194,0.170495
8,Male,5.0,15.0,not_pregnant,0.227134,0.211917,0.202795,0.189340,0.168813
9,Male,15.0,30.0,not_pregnant,0.194480,0.197231,0.196587,0.211924,0.199779


In [26]:
wealth_quintile_probabilities = (
    map_to_population_age_groups(wealth_quintile_probabilities[wealth_quintile_probabilities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))
        .set_index(["sex", "age_start", "age_end"])
)
wealth_quintile_probabilities.columns.name = 'wealth_quintile'
wealth_quintile_probabilities = wealth_quintile_probabilities.stack()
wealth_quintile_probabilities

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    lowest             0.220707
                               second             0.219801
                               middle             0.207518
                               fourth             0.182416
                               highest            0.169557
                                                    ...   
Male    95.0       125.000000  lowest             0.209864
                               second             0.183658
                               middle             0.188784
                               fourth             0.202148
                               highest            0.215546
Length: 250, dtype: float64

In [27]:
assert np.allclose(wealth_quintile_probabilities.groupby(["sex", "age_start", "age_end"]).sum(), 1.0)

In [28]:
def distribute_by_disparities(df, disparities):
    pre_disparity_groups = df.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in df.index.names if c != 'wealth_quintile']).sum()
    print('Before distributing by disparities:')
    display(pre_disparity_groups)

    df = df.mul(disparities, axis=0)

    scale_factor = pre_disparity_groups / df.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in df.index.names if c != 'wealth_quintile']).sum()
    print(f'Scale factor: {scale_factor}')

    df = df * scale_factor

    assert np.allclose(
        df.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in df.index.names if c != 'wealth_quintile']).sum(),
        pre_disparity_groups,
    )

    return df

In [29]:
hgb_mean = distribute_by_disparities(hgb_mean, hemoglobin_mean_disparities)

Before distributing by disparities:


draw_0      draw_1      draw_2      draw_3  \
sex    age_start age_end                                                      
Female 0.000000  0.019178    140.203331  139.700680  137.486100  143.738463   
       0.019178  0.076712    121.992596  122.837138  123.877519  120.604868   
       0.076712  0.500000    104.152465  104.343166  100.933682  103.672346   
       0.500000  1.000000    100.148802   99.844921   99.645127  100.145850   
       1.000000  2.000000    100.955723  100.339208  102.148597  101.234802   
       2.000000  5.000000    105.233662  104.333100  102.835176  105.304765   
       5.000000  10.000000   119.455834  112.957619  115.682929  114.256919   
       10.000000 15.000000   119.312919  106.273992  111.855854  122.097443   
       15.000000 20.000000   114.861226  117.794304  116.359602  117.648438   
       20.000000 25.000000   116.232257  117.251310  116.393998  114.894111   
       25.000000 30.000000   116.082332  114.626063  116.408492  116.811736   
       30.000000 35.000000   115.103530  115.163515  116.767807  116.269128   
       35.000000 40.000000   115.986054  113.528076  114.852973  114.288218   
       40.000000 45.000000   115.257500  117.940598  117.278455  116.853328   
       45.000000 50.000000   118.568757  119.048450  117.278407  117.835148   
       50.000000 55.000000   115.669641  121.096756  111.909301  123.510802   
       55.000000 60.000000   116.888277  111.934668  117.516013  116.335911   
       60.000000 65.000000   117.073529  120.448205  112.138824  119.361783   
       65.000000 70.000000   120.936133  131.305616  125.299052  121.720141   
       70.000000 75.000000   120.728706  120.069195  124.432953  117.908148   
       75.000000 80.000000   126.020295  126.733554  121.556975  127.985198   
       80.000000 85.000000   126.158836  126.248607  116.366249  123.473284   
       85.000000 90.000000   118.780034  118.418548  114.061451  115.310723   
       90.000000 95.000000   109.477010  104.789317  110.610979  108.944244   
       95.000000 125.000000  102.971577  100.970692   99.657313   98.358658   
Male   0.000000  0.019178    139.655000  152.652888  141.405358  135.236759   
       0.019178  0.076712    111.875330  122.227176  119.526917  118.369764   
       0.076712  0.500000     94.267052   98.335982   96.967303   98.720429   
       0.500000  1.000000    100.092403   98.678200   93.913129   99.908232   
       1.000000  2.000000    100.226883   94.470196   98.331789   95.255325   
       2.000000  5.000000    101.069093  107.942797  103.555610  102.374253   
       5.000000  10.000000   119.838610  110.970261  109.073412  112.143982   
       10.000000 15.000000   129.595152  119.217013  127.657332  129.642930   
       15.000000 20.000000   129.129905  144.220990  141.441577  142.995724   
       20.000000 25.000000   151.050744  131.461208  146.612140  132.307327   
       25.000000 30.000000   132.425569  149.591271  142.560174  135.773968   
       30.000000 35.000000   140.605288  141.846950  135.968545  160.662459   
       35.000000 40.000000   138.202586  145.354328  144.617087  133.079794   
       40.000000 45.000000   141.806468  145.535717  136.109962  148.130476   
       45.000000 50.000000   133.416092  136.101393  150.337119  141.486861   
       50.000000 55.000000   144.695059  135.210554  139.837715  139.728713   
       55.000000 60.000000   138.257682  142.365263  142.266348  143.671582   
       60.000000 65.000000   137.633530  133.990371  143.313102  140.484227   
       65.000000 70.000000   143.947810  146.078428  140.887869  134.047618   
       70.000000 75.000000   141.462864  144.115084  130.980064  145.225366   
       75.000000 80.000000   125.293565  127.635808  121.016674  127.115747   
       80.000000 85.000000   136.438006  140.853010  131.367813  139.943361   
       85.000000 90.000000   133.029351  120.328832  136.611180  126.391697   
       90.000000 95.000000   116.866389  104.222078  111.737153  113.207507   
    

Scale factor:                                draw_0    draw_1    draw_2    draw_3    draw_4  \
sex    age_start age_end                                                        
Female 0.000000  0.019178    0.009819  0.009819  0.009819  0.009819  0.009819   
       0.019178  0.076712    0.009819  0.009819  0.009819  0.009819  0.009819   
       0.076712  0.500000    0.009819  0.009819  0.009819  0.009819  0.009819   
       0.500000  1.000000    0.009819  0.009819  0.009819  0.009819  0.009819   
       1.000000  2.000000    0.009819  0.009819  0.009819  0.009819  0.009819   
       2.000000  5.000000    0.009819  0.009819  0.009819  0.009819  0.009819   
       5.000000  10.000000   0.009182  0.009182  0.009182  0.009182  0.009182   
       10.000000 15.000000   0.009182  0.009182  0.009182  0.009182  0.009182   
       15.000000 20.000000   0.008605  0.008605  0.008605  0.008605  0.008605   
       20.000000 25.000000   0.008605  0.008605  0.008605  0.008605  0.008605   
       25.0000

In [30]:
hgb_mean.columns.name = "draw"
hgb_mean = hgb_mean.stack().rename("mean")

In [31]:
hgb_sd = get_modelable_entity_draws(me_ids["hemoglobin_sd"], location)
hgb_sd

draw_0     draw_1     draw_2     draw_3  \
sex    age_start age_end                                                  
Female 0.000000  0.019178    26.026960  23.907182  16.403169  18.715768   
       0.019178  0.076712    29.653984  25.132357  26.064191  35.471576   
       0.076712  0.500000    22.493218  16.259081  17.452182  15.476258   
       0.500000  1.000000    15.088749  16.763458  14.467460  14.968448   
       1.000000  2.000000    15.397715  16.467520  16.931422  15.316047   
       2.000000  5.000000    15.553860  13.236368  15.314247  15.205571   
       5.000000  10.000000   24.246181   8.464774  13.511319  12.174676   
       10.000000 15.000000   20.372010  17.600339  15.670304  18.874888   
       15.000000 20.000000   13.849848  15.303821  13.412809  15.645587   
       20.000000 25.000000   14.161841  13.816199  16.113706  13.064190   
       25.000000 30.000000   13.951453  12.087523  13.960027  15.747683   
       30.000000 35.000000   15.124416  13.912934  14.508748  14.284332   
       35.000000 40.000000   14.537795  12.796153  11.977138  12.619102   
       40.000000 45.000000   15.511511  15.320203  13.814969  15.477653   
       45.000000 50.000000   14.740887  16.461188  14.795978  14.266428   
       50.000000 55.000000   12.005022  19.946933  14.359695  22.342577   
       55.000000 60.000000   15.457708   9.403534  17.510417  14.185617   
       60.000000 65.000000   17.870134  23.535264  15.190874  19.820442   
       65.000000 70.000000   18.274735  38.392728  27.115780  22.460828   
       70.000000 75.000000   14.297770  12.642415  17.848045  11.580773   
       75.000000 80.000000   15.552421  25.187471   9.408474  17.651964   
       80.000000 85.000000   27.470516  25.512507  18.952878  25.817333   
       85.000000 90.000000   18.088001  20.580833   5.178689  16.104535   
       90.000000 95.000000   12.785354  20.398206  15.979831  13.495828   
       95.000000 125.000000  23.461384  26.340875  35.452001  34.646552   
Male   0.000000  0.019178     8.741365   8.444370  10.058485   9.269974   
       0.019178  0.076712    45.945438  28.232303  33.884957  25.657119   
       0.076712  0.500000    10.834561  14.464839  16.716692  13.781233   
       0.500000  1.000000    17.866906  15.667957  15.926898  17.670570   
       1.000000  2.000000    17.452899  16.689185  13.466597  16.075586   
       2.000000  5.000000    11.124414  20.798399  13.040816  21.819873   
       5.000000  10.000000   15.571013  12.788589  13.163620  10.470561   
       10.000000 15.000000   20.593272   7.675260  11.134538  13.284772   
       15.000000 20.000000   11.081504  22.556104  22.074298  21.838462   
       20.000000 25.000000   18.281512   7.567572  14.716066   9.559315   
       25.000000 30.000000    9.776189  17.322972  12.849834   7.167079   
       30.000000 35.000000   12.832856  14.604296   8.867764  27.680958   
       35.000000 40.000000    7.747828  16.962800  16.855032  10.901524   
       40.000000 45.000000   11.482526  14.684343   8.398772  22.837871   
       45.000000 50.000000   12.619049  13.035039  20.963212  15.980802   
       50.000000 55.000000   15.864704  16.779504  15.424543  21.599276   
       55.000000 60.000000   10.580076  27.129333  19.432159  17.335354   
       60.000000 65.000000   15.722404  12.879010  20.046828  24.112250   
       65.000000 70.000000   21.271206  16.053375  16.248970  17.207534   
       70.000000 75.000000   24.062525  24.232873  13.549617  18.340322   
       75.000000 80.000000    9.509731  13.080280   8.034360   9.796766   
       80.000000 85.000000   20.360018  15.849824  13.075308  21.846383   
       85.000000 90.000000   30.679256  12.187568  24.214234  24.380637   
       90.000000 95.000000   12.432987  22.692279   9.217534   6.586342   
       95.000000 125.000000  25.841204  20.160131  32.834640  21.792155   

                                draw_4     draw_5     draw_6     draw_7  \
sex    age_start age_end                                

In [32]:
hemoglobin_sd_disparities = pd.read_csv(f'../0100_data_prep/results/hemoglobin/sd_disparities/{location}.csv')
hemoglobin_sd_disparities = (
    map_to_population_age_groups(hemoglobin_sd_disparities[hemoglobin_sd_disparities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))
        .set_index(["sex", "age_start", "age_end", "wealth_quintile"]).value
)
hemoglobin_sd_disparities

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    lowest             16.607283
                               second             15.898899
                               middle             15.268520
                               fourth             14.286274
                               highest            13.194608
                                                    ...    
Male    95.0       125.000000  lowest             15.148809
                               second             14.920351
                               middle             15.108541
                               fourth             14.792239
                               highest            14.222341
Name: value, Length: 250, dtype: float64

In [33]:
hgb_sd = distribute_by_disparities(hgb_sd, hemoglobin_sd_disparities)
hgb_sd

Before distributing by disparities:


draw_0     draw_1     draw_2     draw_3  \
sex    age_start age_end                                                  
Female 0.000000  0.019178    26.026960  23.907182  16.403169  18.715768   
       0.019178  0.076712    29.653984  25.132357  26.064191  35.471576   
       0.076712  0.500000    22.493218  16.259081  17.452182  15.476258   
       0.500000  1.000000    15.088749  16.763458  14.467460  14.968448   
       1.000000  2.000000    15.397715  16.467520  16.931422  15.316047   
       2.000000  5.000000    15.553860  13.236368  15.314247  15.205571   
       5.000000  10.000000   24.246181   8.464774  13.511319  12.174676   
       10.000000 15.000000   20.372010  17.600339  15.670304  18.874888   
       15.000000 20.000000   13.849848  15.303821  13.412809  15.645587   
       20.000000 25.000000   14.161841  13.816199  16.113706  13.064190   
       25.000000 30.000000   13.951453  12.087523  13.960027  15.747683   
       30.000000 35.000000   15.124416  13.912934  14.508748  14.284332   
       35.000000 40.000000   14.537795  12.796153  11.977138  12.619102   
       40.000000 45.000000   15.511511  15.320203  13.814969  15.477653   
       45.000000 50.000000   14.740887  16.461188  14.795978  14.266428   
       50.000000 55.000000   12.005022  19.946933  14.359695  22.342577   
       55.000000 60.000000   15.457708   9.403534  17.510417  14.185617   
       60.000000 65.000000   17.870134  23.535264  15.190874  19.820442   
       65.000000 70.000000   18.274735  38.392728  27.115780  22.460828   
       70.000000 75.000000   14.297770  12.642415  17.848045  11.580773   
       75.000000 80.000000   15.552421  25.187471   9.408474  17.651964   
       80.000000 85.000000   27.470516  25.512507  18.952878  25.817333   
       85.000000 90.000000   18.088001  20.580833   5.178689  16.104535   
       90.000000 95.000000   12.785354  20.398206  15.979831  13.495828   
       95.000000 125.000000  23.461384  26.340875  35.452001  34.646552   
Male   0.000000  0.019178     8.741365   8.444370  10.058485   9.269974   
       0.019178  0.076712    45.945438  28.232303  33.884957  25.657119   
       0.076712  0.500000    10.834561  14.464839  16.716692  13.781233   
       0.500000  1.000000    17.866906  15.667957  15.926898  17.670570   
       1.000000  2.000000    17.452899  16.689185  13.466597  16.075586   
       2.000000  5.000000    11.124414  20.798399  13.040816  21.819873   
       5.000000  10.000000   15.571013  12.788589  13.163620  10.470561   
       10.000000 15.000000   20.593272   7.675260  11.134538  13.284772   
       15.000000 20.000000   11.081504  22.556104  22.074298  21.838462   
       20.000000 25.000000   18.281512   7.567572  14.716066   9.559315   
       25.000000 30.000000    9.776189  17.322972  12.849834   7.167079   
       30.000000 35.000000   12.832856  14.604296   8.867764  27.680958   
       35.000000 40.000000    7.747828  16.962800  16.855032  10.901524   
       40.000000 45.000000   11.482526  14.684343   8.398772  22.837871   
       45.000000 50.000000   12.619049  13.035039  20.963212  15.980802   
       50.000000 55.000000   15.864704  16.779504  15.424543  21.599276   
       55.000000 60.000000   10.580076  27.129333  19.432159  17.335354   
       60.000000 65.000000   15.722404  12.879010  20.046828  24.112250   
       65.000000 70.000000   21.271206  16.053375  16.248970  17.207534   
       70.000000 75.000000   24.062525  24.232873  13.549617  18.340322   
       75.000000 80.000000    9.509731  13.080280   8.034360   9.796766   
       80.000000 85.000000   20.360018  15.849824  13.075308  21.846383   
       85.000000 90.000000   30.679256  12.187568  24.214234  24.380637   
       90.000000 95.000000   12.432987  22.692279   9.217534   6.586342   
       95.000000 125.000000  25.841204  20.160131  32.834640  21.792155   

                                draw_4     draw_5     draw_6     draw_7  \
sex    age_start age_end                                

Scale factor:                                draw_0    draw_1    draw_2    draw_3    draw_4  \
sex    age_start age_end                                                        
Female 0.000000  0.019178    0.065912  0.065912  0.065912  0.065912  0.065912   
       0.019178  0.076712    0.065912  0.065912  0.065912  0.065912  0.065912   
       0.076712  0.500000    0.065912  0.065912  0.065912  0.065912  0.065912   
       0.500000  1.000000    0.065912  0.065912  0.065912  0.065912  0.065912   
       1.000000  2.000000    0.065912  0.065912  0.065912  0.065912  0.065912   
       2.000000  5.000000    0.065912  0.065912  0.065912  0.065912  0.065912   
       5.000000  10.000000   0.066637  0.066637  0.066637  0.066637  0.066637   
       10.000000 15.000000   0.066637  0.066637  0.066637  0.066637  0.066637   
       15.000000 20.000000   0.067493  0.067493  0.067493  0.067493  0.067493   
       20.000000 25.000000   0.067493  0.067493  0.067493  0.067493  0.067493   
       25.0000

draw_0     draw_1     draw_2  \
sex    age_start age_end    wealth_quintile                                    
Female 0.0       0.019178   lowest           28.489640  26.169288  17.955243   
                            second           27.274413  25.053035  17.189361   
                            middle           26.193003  24.059702  16.507816   
                            fourth           24.507970  22.511907  15.445845   
                            highest          22.635227  20.791691  14.265572   
...                                                ...        ...        ...   
Male   95.0      125.000000 lowest           26.401227  20.597035  33.546224   
                            second           26.003071  20.286412  33.040313   
                            middle           26.331048  20.542285  33.457051   
                            fourth           25.779798  20.112225  32.756616   
                            highest          24.786586  19.337366  31.494610   

                                                draw_3     draw_4     draw_5  \
sex    age_start age_end    wealth_quintile                                    
Female 0.0       0.019178   lowest           20.486661  18.648197  23.485834   
                            second           19.612801  17.852757  22.484045   
                            middle           18.835169  17.144909  21.592570   
                            fourth           17.623475  16.041953  20.203489   
                            highest          16.276802  14.816129  18.659667   
...                                                ...        ...        ...   
Male   95.0      125.000000 lowest           22.264429  16.191703  18.636087   
                            second           21.928659  15.947516  18.355036   
                            middle           22.205246  16.148662  18.586548   
                            fourth           21.740371  15.810584  18.197433   
                            highest          20.902785  15.201454  17.496344   

                                                draw_6     draw_7     draw_8  \
sex    age_start age_end    wealth_quintile                                    
Female 0.0       0.019178   lowest           14.589976  29.735760  35.667006   
                            second           13.967640  28.467380  34.145628   
                            middle           13.413834  27.338670  32.791780   
                            fourth           12.550903  25.579935  30.682238   
                            highest          11.591843  23.625279  28.337697   
...                                                ...        ...        ...   
Male   95.0      125.000000 lowest           21.172638  11.791330  19.286321   
                            second           20.853334  11.613505  18.995465   
                            middle           21.116357  11.759986  19.235054   
                            fourth           20.674279  11.513787  18.832362   
                            highest          19.877766  11.070198  18.106812   

                                                draw_9    draw_10    draw_11  \
sex    age_start age_end    wealth_quintile                                    
Female 0.0       0.019178   lowest            8.776604  14.442309  16.832872   
                            second            8.402238  13.826271  16.114865   
                            middle            8.069095  13.278070  15.475923   
                            fourth            7.549999  12.423873  14.480335   
                            highest           6.973076  11.474520  13.373840   
...                                                ...        ...        ...   
Male   95.0      125.000000 lowest           10.931228  52.593354  16.659904   
                            second           10.766375  51.800194  16.408656   
                            middle           10.902171  52.453551  16.615618   
                            fourth           

In [34]:
hgb_sd.columns.name = "draw"
hgb_sd = hgb_sd.stack().rename("sd")
hgb_sd

sex     age_start  age_end     wealth_quintile  draw    
Female  0.0        0.019178    lowest           draw_0      28.489640
                                                draw_1      26.169288
                                                draw_2      17.955243
                                                draw_3      20.486661
                                                draw_4      18.648197
                                                              ...    
Male    95.0       125.000000  highest          draw_495    33.780207
                                                draw_496    18.376878
                                                draw_497    22.113952
                                                draw_498    35.369247
                                                draw_499    16.712756
Name: sd, Length: 125000, dtype: float64

## Effect size and adjustment for iron responsiveness

We assume that our overall effect size is composed of two parts:
some people respond to iron with a constant shift (no individual heterogeneity)
and other people are "not responsive" and their hemoglobin doesn't
change at all.

This is similar to how GBD models the iron deficiency risk factor.
I believe we got the lists of sequelae below from them.

As a rough approximation, we assume that our mean difference value
(from the literature) was from a population that had the global prevalence
split between iron-responsive and non-iron-responsive.

**Note: We assume everyone who is not anemic is iron-responsive.**

In [35]:
fortification_hemoglobin_mean_difference = (
    pd.read_csv('../0100_data_prep/results/iron/fortification_hemoglobin_effects.csv')
        .set_index('vehicle_name').value
        .loc[vehicle]
)
fortification_hemoglobin_mean_difference

4.2

In [36]:
# Cleaned this up from https://github.com/ihmeuw/vivarium_research_lsff/blob/1cb465a752d299401ae366db537dc8d557162184/multiplication_models/iron_model_U5.ipynb,
# but have not checked it in extreme detail.
iron_responsive_anemia_sequelae = [
    gbd_mapping.sequelae.mild_anemia_due_to_schistosomiasis,
    gbd_mapping.sequelae.moderate_anemia_due_to_schistosomiasis,
    gbd_mapping.sequelae.severe_anemia_due_to_schistosomiasis,
    gbd_mapping.sequelae.mild_anemia_due_to_hookworm_disease,
    gbd_mapping.sequelae.moderate_anemia_due_to_hookworm_disease,
    gbd_mapping.sequelae.severe_anemia_due_to_hookworm_disease,
    gbd_mapping.sequelae.mild_anemia_due_to_other_neglected_tropical_diseases,
    gbd_mapping.sequelae.moderate_anemia_due_to_other_neglected_tropical_diseases,
    gbd_mapping.sequelae.severe_anemia_due_to_other_neglected_tropical_diseases,
    gbd_mapping.sequelae.mild_anemia_due_to_maternal_hemorrhage,
    gbd_mapping.sequelae.moderate_anemia_due_to_maternal_hemorrhage,
    gbd_mapping.sequelae.severe_anemia_due_to_maternal_hemorrhage,
    gbd_mapping.sequelae.mild_iron_deficiency_anemia,
    gbd_mapping.sequelae.moderate_iron_deficiency_anemia,
    gbd_mapping.sequelae.severe_iron_deficiency_anemia,
    gbd_mapping.sequelae.mild_anemia_due_to_other_infectious_diseases,
    gbd_mapping.sequelae.moderate_anemia_due_to_other_infectious_diseases,
    gbd_mapping.sequelae.severe_anemia_due_to_other_infectious_diseases,
    gbd_mapping.sequelae.menstrual_disorders_with_mild_anemia,
    gbd_mapping.sequelae.menstrual_disorders_with_moderate_anemia,
    gbd_mapping.sequelae.menstrual_disorders_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.uterine_fibroids_symptomatic_with_mild_anemia,
    gbd_mapping.sequelae.uterine_fibroids_symptomatic_with_moderate_anemia,
    gbd_mapping.sequelae.uterine_fibroids_symptomatic_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_pud_with_mild_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_pud_with_mild_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_pud_with_moderate_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_pud_with_moderate_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_pud_with_severe_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_pud_with_severe_anemia,
    gbd_mapping.sequelae.asymptomatic_pud_with_mild_anemia,
    gbd_mapping.sequelae.asymptomatic_pud_with_moderate_anemia,
    gbd_mapping.sequelae.asymptomatic_pud_with_severe_anemia,
    gbd_mapping.sequelae.mildy_symptomatic_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.mildy_symptomatic_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.asymptomatic_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.asymptomatic_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.asymptomatic_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_2_diabetes_mellitus_with_mdoerate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_mdoerate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.vitamin_a_deficiency_with_mild_anemia,
    gbd_mapping.sequelae.vitamin_a_deficiency_with_moderate_anemia,
    gbd_mapping.sequelae.vitamin_a_deficiency_with_severe_anemia,
    gbd_mapping.sequelae.ulcerative_colitis_with_mild_anemia,
    gbd_mapping.sequelae.ulcerative_colitis_with_moderate_anemia,
    gbd_mapping.sequelae.ulcerative_colitis_with_severe_anemia,
    gbd_mapping.sequelae.crohns_disease_with_mild_anemia,
    gbd_mapping.sequelae.crohns_disease_with_moderate_anemia,
    gbd_mapping.sequelae.crohns_disease_with_severe_anemia,
    gbd_mapping.sequelae.complicated_pud_with_mild_anemia,
    gbd_mapping.sequelae.complicated_pud_with_moderate_anemia,
    gbd_mapping.sequelae.complicated_pud_with_severe_anemia,
    gbd_mapping.sequelae.complicated_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.complicated_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.complicated_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_pud_with_mild_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_pud_with_moderate_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_pud_with_severe_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_2_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_b_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_b_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_b_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_c_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_c_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_c_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_alcohol_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_alcohol_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_alcohol_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_other_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_other_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_other_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_nash_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_nash_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_nash_decompensated_with_severe_anemia,
]

In [37]:
non_iron_responsive_anemia_sequelae = [
    gbd_mapping.sequelae.mild_anemia_due_to_other_hemoglobinopathies_and_hemolytic_anemias,
    gbd_mapping.sequelae.moderate_anemia_due_to_other_hemoglobinopathies_and_hemolytic_anemias,
    gbd_mapping.sequelae.severe_anemia_due_to_other_hemoglobinopathies_and_hemolytic_anemias,
    gbd_mapping.sequelae.mild_anemia_due_to_b_thalassemia_trait,
    gbd_mapping.sequelae.moderate_anemia_due_to_b_thalassemia_trait,
    gbd_mapping.sequelae.severe_anemia_due_to_b_thalassemia_trait,
    gbd_mapping.sequelae.mild_anemia_due_to_hemoglobin_e_trait,
    gbd_mapping.sequelae.moderate_anemia_due_to_hemoglobin_e_trait,
    gbd_mapping.sequelae.severe_anemia_due_to_hemoglobin_e_trait,
    gbd_mapping.sequelae.mild_anemia_due_to_sickle_cell_trait,
    gbd_mapping.sequelae.moderate_anemia_due_to_sickle_cell_trait,
    gbd_mapping.sequelae.severe_anemia_due_to_sickle_cell_trait,
    gbd_mapping.sequelae.hemizygous_g6pd_deficiency_with_mild_anemia,
    gbd_mapping.sequelae.hemizygous_g6pd_deficiency_with_moderate_anemia,
    gbd_mapping.sequelae.hemizygous_g6pd_deficiency_with_severe_anemia,
    gbd_mapping.sequelae.mild_anemia_due_to_malaria_parasitemia_pfpr,
    gbd_mapping.sequelae.moderate_anemia_due_to_malaria_parasitemia_pfpr,
    gbd_mapping.sequelae.severe_anemia_due_to_malaria_parasitemia_pfpr,
    gbd_mapping.sequelae.mild_anemia_due_to_homozygous_sickle_cell_and_severe_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.moderate_anemia_due_to_homozygous_sickle_cell_and_severe_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.severe_anemia_due_to_homozygous_sickle_cell_and_severe_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.mild_anemia_due_to_hemoglobin_sc_disease,
    gbd_mapping.sequelae.moderate_anemia_due_to_hemoglobin_sc_disease,
    gbd_mapping.sequelae.severe_anemia_due_to_hemoglobin_sc_disease,
    gbd_mapping.sequelae.mild_anemia_due_to_mild_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.moderate_anemia_due_to_mild_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.severe_anemia_due_to_mild_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.severe_malaria_with_mild_anemia,
    gbd_mapping.sequelae.severe_malaria_with_moderate_anemia,
    gbd_mapping.sequelae.severe_malaria_with_severe_anemia,
    gbd_mapping.sequelae.mild_malaria_with_mild_anemia,
    gbd_mapping.sequelae.mild_malaria_with_moderate_anemia,
    gbd_mapping.sequelae.mild_malaria_with_severe_anemia,
    gbd_mapping.sequelae.moderate_malaria_with_mild_anemia,
    gbd_mapping.sequelae.moderate_malaria_with_moderate_anemia,
    gbd_mapping.sequelae.moderate_malaria_with_severe_anemia,
    gbd_mapping.sequelae.early_hiv_with_mild_anemia,
    gbd_mapping.sequelae.early_hiv_with_moderate_anemia,
    gbd_mapping.sequelae.early_hiv_with_severe_anemia,
    gbd_mapping.sequelae.symptomatic_hiv_with_mild_anemia,
    gbd_mapping.sequelae.symptomatic_hiv_with_moderate_anemia,
    gbd_mapping.sequelae.symptomatic_hiv_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_with_antiretroviral_treatment_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_with_antiretroviral_treatment_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_with_antiretroviral_treatment_with_severe_anemia,
    gbd_mapping.sequelae.aids_with_mild_anemia,
    gbd_mapping.sequelae.aids_with_moderate_anemia,
    gbd_mapping.sequelae.aids_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_drug_susceptible_tuberculosis_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_drug_susceptible_tuberculosis_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_drug_susceptible_tuberculosis_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_multidrug_resistant_tuberculosis_without_extensive_drug_resistance_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_multidrug_resistant_tuberculosis_without_extensive_drug_resistance_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_multidrug_resistant_tuberculosis_without_extensive_drug_resistance_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_extensively_drug_resistant_tuberculosis_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_extensively_drug_resistant_tuberculosis_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_extensively_drug_resistant_tuberculosis_with_severe_anemia,
    gbd_mapping.sequelae.mild_anemia_due_to_malaria_vivax_pvpr,
    gbd_mapping.sequelae.moderate_anemia_due_to_malaria_vivax_pvpr,
    gbd_mapping.sequelae.severe_anemia_due_to_malaria_vivax_pvpr,
]

In [38]:
len(iron_responsive_anemia_sequelae)

138

In [39]:
len(non_iron_responsive_anemia_sequelae)

60

In [40]:
def pull_sequelae_prevalence(location, sequelae):
    result = 0
    # There are tons of validation warnings -- look into these more?
    loguru.logger.disable("vivarium_inputs.validation.raw")
    for sequela in sequelae:
        try:
            sequela_prevalence = vivarium_inputs.get_measure(sequela, "prevalence", location.title()).droplevel(["location"])
        # There are even some errors, caused by all-zero values
        except DataDoesNotExistError as e:
            assert 'zero' in str(e)
            continue
        except DataAbnormalError as e:
            assert 'zero' in str(e)
            continue

        # AFAIK these are not mutually exclusive; standard GBD assumption is independence
        result += sequela_prevalence * (1 - result)
    
    loguru.logger.enable("vivarium_inputs.validation.raw")
    
    return result

In [41]:
global_population = vivarium_inputs.get_population_structure("Global").droplevel("location").value
global_population

sex     age_start  age_end     year_start  year_end
Female  0.000000   0.019178    2021        2022        1.183499e+06
        0.019178   0.076712    2021        2022        3.525317e+06
        0.076712   0.500000    2021        2022        2.598414e+07
        0.500000   1.000000    2021        2022        3.054267e+07
        1.000000   2.000000    2021        2022        6.209731e+07
        2.000000   5.000000    2021        2022        1.948595e+08
        5.000000   10.000000   2021        2022        3.325771e+08
        10.000000  15.000000   2021        2022        3.229261e+08
        15.000000  20.000000   2021        2022        3.036500e+08
        20.000000  25.000000   2021        2022        2.937520e+08
        25.000000  30.000000   2021        2022        2.909873e+08
        30.000000  35.000000   2021        2022        2.989309e+08
        35.000000  40.000000   2021        2022        2.778027e+08
        40.000000  45.000000   2021        2022        2.480899e

In [42]:
global_non_responsive = pull_sequelae_prevalence("Global", non_iron_responsive_anemia_sequelae)
global_non_responsive

draw_0    draw_1    draw_2  \
sex    age_start age_end    year_start year_end                                 
Female 0.000000  0.019178   2021       2022      0.055640  0.056572  0.054542   
       0.019178  0.076712   2021       2022      0.043064  0.043721  0.049355   
       0.076712  0.500000   2021       2022      0.048061  0.050737  0.047986   
       0.500000  1.000000   2021       2022      0.057733  0.055035  0.056787   
       1.000000  2.000000   2021       2022      0.053650  0.050164  0.050987   
       2.000000  5.000000   2021       2022      0.043599  0.045122  0.043894   
       5.000000  10.000000  2021       2022      0.050148  0.045555  0.049773   
       10.000000 15.000000  2021       2022      0.039617  0.041705  0.039306   
       15.000000 20.000000  2021       2022      0.050980  0.051745  0.050868   
       20.000000 25.000000  2021       2022      0.054177  0.052733  0.053112   
       25.000000 30.000000  2021       2022      0.047947  0.047760  0.047118   
       30.000000 35.000000  2021       2022      0.046199  0.045096  0.045326   
       35.000000 40.000000  2021       2022      0.042182  0.044672  0.045842   
       40.000000 45.000000  2021       2022      0.043585  0.044319  0.041669   
       45.000000 50.000000  2021       2022      0.035960  0.038896  0.037265   
       50.000000 55.000000  2021       2022      0.036418  0.034481  0.036316   
       55.000000 60.000000  2021       2022      0.030355  0.031916  0.030824   
       60.000000 65.000000  2021       2022      0.041190  0.041390  0.041520   
       65.000000 70.000000  2021       2022      0.041969  0.041562  0.043108   
       70.000000 75.000000  2021       2022      0.038342  0.038004  0.036470   
       75.000000 80.000000  2021       2022      0.041209  0.037947  0.038835   
       80.000000 85.000000  2021       2022      0.036829  0.036521  0.035043   
       85.000000 90.000000  2021       2022      0.034473  0.034188  0.034005   
       90.000000 95.000000  2021       2022      0.055046  0.053303  0.053106   
       95.000000 125.000000 2021       2022      0.075064  0.063504  0.067947   
Male   0.000000  0.019178   2021       2022      0.061047  0.056001  0.058363   
       0.019178  0.076712   2021       2022      0.050700  0.045030  0.042907   
       0.076712  0.500000   2021       2022      0.065543  0.054614  0.057981   
       0.500000  1.000000   2021       2022      0.058564  0.061917  0.059273   
       1.000000  2.000000   2021       2022      0.056070  0.056654  0.055482   
       2.000000  5.000000   2021       2022      0.049919  0.049917  0.047593   
       5.000000  10.000000  2021       2022      0.048810  0.051523  0.053304   
       10.000000 15.000000  2021       2022      0.028810  0.029847  0.028401   
       15.000000 20.000000  2021       2022      0.031138  0.033774  0.035536   
       20.000000 25.000000  2021       2022      0.021930  0.021960  0.023117   
       25.000000 30.000000  2021       2022      0.019615  0.018715  0.018056   
       30.000000 35.000000  2021       2022      0.019797  0.020312  0.018503   
       35.000000 40.000000  2021       2022      0.020464  0.019792  0.019279   
       40.000000 45.000000  2021       2022      0.021660  0.020551  0.020673   
       45.000000 50.000000  2021       2022      0.023092  0.019379  0.020213   
       50.000000 55.000000  2021       2022      0.022401  0.022282  0.023421   
       55.000000 60.000000  2021       2022      0.026887  0.027794  0.028187   
       60.000000 65.000000  2021       2022      0.031909  0.031755  0.031248   
       65.000000 70.000000  2021       2022      0.032452  0.033886  0.033783   
       70.000000 75.000000  2021       2022      0.038492  0.038466  0.034610   
       75.000000 80.000000  2021       2022      0.044614  0.039762  0.040172   
       80.000000 85.000000  2021       2022      0.046900  0.040499  0.040998   
       85.000000 90.000000  2021       2022      0.052768  0.046749  0.046760   
 

In [43]:
global_non_responsive_aggregated = global_non_responsive.mul(global_population, axis=0).sum() / global_population.sum()
global_non_responsive_aggregated.index.name = "draw"
global_non_responsive_aggregated

draw
draw_0      0.037053
draw_1      0.037039
draw_2      0.036969
draw_3      0.037445
draw_4      0.038785
              ...   
draw_495    0.038454
draw_496    0.037231
draw_497    0.037608
draw_498    0.036549
draw_499    0.038063
Length: 500, dtype: float64

In [44]:
global_non_responsive_aggregated.describe()

count    500.000000
mean       0.037534
std        0.000972
min        0.035248
25%        0.036898
50%        0.037495
75%        0.038176
max        0.041075
dtype: float64

In [45]:
# mean difference observed = 0 * non-responsive + mean_difference_responsive * (1 - non-responsive)
# Assume observed in total population (some studies in the meta-analysis only included children,
# but we are applying the effect to total population anyway)
hemoglobin_effect_among_responsive = fortification_hemoglobin_mean_difference / (1 - global_non_responsive_aggregated)
hemoglobin_effect_among_responsive

draw
draw_0      4.361610
draw_1      4.361547
draw_2      4.361229
draw_3      4.363386
draw_4      4.369470
              ...   
draw_495    4.367965
draw_496    4.362416
draw_497    4.364126
draw_498    4.359328
draw_499    4.366189
Length: 500, dtype: float64

## Iron-responsiveness in population of interest

Note: unlike the previous, we do this *as a fraction of the anemic population*.
That is because it is only the anemic population where our shifting intervention
makes a difference (in prevalence/YLDs).

In [46]:
iron_responsive_prevalence = pull_sequelae_prevalence(location, iron_responsive_anemia_sequelae)
iron_responsive_prevalence

draw_0    draw_1    draw_2  \
sex    age_start age_end    year_start year_end                                 
Female 0.000000  0.019178   2021       2022      0.630256  0.552243  0.566283   
       0.019178  0.076712   2021       2022      0.448395  0.442822  0.454978   
       0.076712  0.500000   2021       2022      0.433914  0.463016  0.470037   
       0.500000  1.000000   2021       2022      0.497132  0.467439  0.499621   
       1.000000  2.000000   2021       2022      0.491708  0.464581  0.459638   
       2.000000  5.000000   2021       2022      0.412388  0.412764  0.400591   
       5.000000  10.000000  2021       2022      0.483957  0.359348  0.442362   
       10.000000 15.000000  2021       2022      0.338579  0.312670  0.313683   
       15.000000 20.000000  2021       2022      0.346955  0.351127  0.375556   
       20.000000 25.000000  2021       2022      0.396640  0.395718  0.397919   
       25.000000 30.000000  2021       2022      0.386243  0.380990  0.368447   
       30.000000 35.000000  2021       2022      0.381479  0.414161  0.420743   
       35.000000 40.000000  2021       2022      0.398081  0.379718  0.381378   
       40.000000 45.000000  2021       2022      0.389609  0.397476  0.376331   
       45.000000 50.000000  2021       2022      0.325443  0.336985  0.326614   
       50.000000 55.000000  2021       2022      0.365623  0.377125  0.335110   
       55.000000 60.000000  2021       2022      0.404799  0.383215  0.384884   
       60.000000 65.000000  2021       2022      0.232927  0.310861  0.281375   
       65.000000 70.000000  2021       2022      0.293502  0.267350  0.248730   
       70.000000 75.000000  2021       2022      0.193271  0.228376  0.265293   
       75.000000 80.000000  2021       2022      0.210077  0.236498  0.293019   
       80.000000 85.000000  2021       2022      0.269934  0.274380  0.301872   
       85.000000 90.000000  2021       2022      0.382655  0.385975  0.333644   
       90.000000 95.000000  2021       2022      0.504216  0.488994  0.531235   
       95.000000 125.000000 2021       2022      0.542416  0.464986  0.471366   
Male   0.000000  0.019178   2021       2022      0.599835  0.636871  0.593881   
       0.019178  0.076712   2021       2022      0.514550  0.486679  0.481996   
       0.076712  0.500000   2021       2022      0.492098  0.513269  0.499941   
       0.500000  1.000000   2021       2022      0.476617  0.530561  0.528565   
       1.000000  2.000000   2021       2022      0.434821  0.508836  0.488606   
       2.000000  5.000000   2021       2022      0.461676  0.398715  0.482157   
       5.000000  10.000000  2021       2022      0.213805  0.345659  0.165647   
       10.000000 15.000000  2021       2022      0.117730  0.179229  0.171587   
       15.000000 20.000000  2021       2022      0.156577  0.322814  0.465596   
       20.000000 25.000000  2021       2022      0.100965  0.106205  0.168078   
       25.000000 30.000000  2021       2022      0.092922  0.097676  0.064137   
       30.000000 35.000000  2021       2022      0.149805  0.155392  0.122008   
       35.000000 40.000000  2021       2022      0.120838  0.114508  0.092046   
       40.000000 45.000000  2021       2022      0.141141  0.095688  0.155755   
       45.000000 50.000000  2021       2022      0.175499  0.137278  0.187488   
       50.000000 55.000000  2021       2022      0.158651  0.278538  0.148858   
       55.000000 60.000000  2021       2022      0.258331  0.159157  0.267639   
       60.000000 65.000000  2021       2022      0.205909  0.173374  0.134717   
       65.000000 70.000000  2021       2022      0.191400  0.218433  0.187046   
       70.000000 75.000000  2021       2022      0.130572  0.291244  0.149748   
       75.000000 80.000000  2021       2022      0.300736  0.260366  0.211547   
       80.000000 85.000000  2021       2022      0.338733  0.365061  0.326679   
       85.000000 90.000000  2021       2022      0.377936  0.310926  0.276739   
 

In [47]:
non_iron_responsive_prevalence = pull_sequelae_prevalence(location, non_iron_responsive_anemia_sequelae)
non_iron_responsive_prevalence

draw_0    draw_1    draw_2  \
sex    age_start age_end    year_start year_end                                 
Female 0.000000  0.019178   2021       2022      0.113243  0.102647  0.131117   
       0.019178  0.076712   2021       2022      0.083052  0.074469  0.096953   
       0.076712  0.500000   2021       2022      0.122898  0.154165  0.143967   
       0.500000  1.000000   2021       2022      0.155177  0.191612  0.186670   
       1.000000  2.000000   2021       2022      0.150334  0.160012  0.148396   
       2.000000  5.000000   2021       2022      0.165317  0.155158  0.142061   
       5.000000  10.000000  2021       2022      0.122023  0.187493  0.239246   
       10.000000 15.000000  2021       2022      0.179526  0.126009  0.132237   
       15.000000 20.000000  2021       2022      0.127837  0.128039  0.156006   
       20.000000 25.000000  2021       2022      0.136359  0.151109  0.156223   
       25.000000 30.000000  2021       2022      0.148853  0.129936  0.149131   
       30.000000 35.000000  2021       2022      0.119783  0.141177  0.134476   
       35.000000 40.000000  2021       2022      0.137657  0.111173  0.121994   
       40.000000 45.000000  2021       2022      0.117908  0.137276  0.126519   
       45.000000 50.000000  2021       2022      0.090217  0.108871  0.100417   
       50.000000 55.000000  2021       2022      0.135154  0.097816  0.077688   
       55.000000 60.000000  2021       2022      0.118294  0.129543  0.090231   
       60.000000 65.000000  2021       2022      0.072133  0.098463  0.100093   
       65.000000 70.000000  2021       2022      0.102507  0.077567  0.086843   
       70.000000 75.000000  2021       2022      0.063607  0.062660  0.070589   
       75.000000 80.000000  2021       2022      0.069206  0.073125  0.116708   
       80.000000 85.000000  2021       2022      0.073740  0.076964  0.079358   
       85.000000 90.000000  2021       2022      0.110161  0.109367  0.099828   
       90.000000 95.000000  2021       2022      0.131630  0.138822  0.150089   
       95.000000 125.000000 2021       2022      0.124839  0.099112  0.098258   
Male   0.000000  0.019178   2021       2022      0.144346  0.100608  0.149151   
       0.019178  0.076712   2021       2022      0.080039  0.062035  0.064115   
       0.076712  0.500000   2021       2022      0.152790  0.146974  0.144892   
       0.500000  1.000000   2021       2022      0.145087  0.177765  0.165804   
       1.000000  2.000000   2021       2022      0.151591  0.206227  0.168827   
       2.000000  5.000000   2021       2022      0.161917  0.131900  0.166681   
       5.000000  10.000000  2021       2022      0.128436  0.213843  0.199769   
       10.000000 15.000000  2021       2022      0.088259  0.083777  0.135801   
       15.000000 20.000000  2021       2022      0.071452  0.133155  0.161292   
       20.000000 25.000000  2021       2022      0.059940  0.070853  0.125009   
       25.000000 30.000000  2021       2022      0.066422  0.043901  0.040308   
       30.000000 35.000000  2021       2022      0.072167  0.084494  0.077521   
       35.000000 40.000000  2021       2022      0.067925  0.047054  0.044467   
       40.000000 45.000000  2021       2022      0.056324  0.047885  0.061609   
       45.000000 50.000000  2021       2022      0.079600  0.049280  0.067731   
       50.000000 55.000000  2021       2022      0.047651  0.087194  0.054428   
       55.000000 60.000000  2021       2022      0.084336  0.088648  0.112104   
       60.000000 65.000000  2021       2022      0.062090  0.072775  0.065757   
       65.000000 70.000000  2021       2022      0.096395  0.062318  0.073333   
       70.000000 75.000000  2021       2022      0.039856  0.108371  0.056037   
       75.000000 80.000000  2021       2022      0.097280  0.080241  0.075393   
       80.000000 85.000000  2021       2022      0.090730  0.102831  0.114650   
       85.000000 90.000000  2021       2022      0.087028  0.077202  0.067757   
 

In [48]:
iron_responsive_proportion = iron_responsive_prevalence / (iron_responsive_prevalence + non_iron_responsive_prevalence)
iron_responsive_proportion

draw_0    draw_1    draw_2  \
sex    age_start age_end    year_start year_end                                 
Female 0.000000  0.019178   2021       2022      0.847689  0.843261  0.811992   
       0.019178  0.076712   2021       2022      0.843726  0.856041  0.824339   
       0.076712  0.500000   2021       2022      0.779283  0.750211  0.765528   
       0.500000  1.000000   2021       2022      0.762111  0.709260  0.728001   
       1.000000  2.000000   2021       2022      0.765850  0.743814  0.755941   
       2.000000  5.000000   2021       2022      0.713838  0.726798  0.738209   
       5.000000  10.000000  2021       2022      0.798635  0.657134  0.648997   
       10.000000 15.000000  2021       2022      0.653495  0.712753  0.703451   
       15.000000 20.000000  2021       2022      0.730751  0.732787  0.706513   
       20.000000 25.000000  2021       2022      0.744167  0.723663  0.718082   
       25.000000 30.000000  2021       2022      0.721820  0.745685  0.711868   
       30.000000 35.000000  2021       2022      0.761037  0.745782  0.757796   
       35.000000 40.000000  2021       2022      0.743051  0.773528  0.757647   
       40.000000 45.000000  2021       2022      0.767677  0.743291  0.748397   
       45.000000 50.000000  2021       2022      0.782955  0.755816  0.764848   
       50.000000 55.000000  2021       2022      0.730112  0.794046  0.811801   
       55.000000 60.000000  2021       2022      0.773856  0.747360  0.810086   
       60.000000 65.000000  2021       2022      0.763546  0.759449  0.737612   
       65.000000 70.000000  2021       2022      0.741149  0.775114  0.741211   
       70.000000 75.000000  2021       2022      0.752384  0.784701  0.789839   
       75.000000 80.000000  2021       2022      0.752200  0.763825  0.715156   
       80.000000 85.000000  2021       2022      0.785436  0.780943  0.791837   
       85.000000 90.000000  2021       2022      0.776467  0.779210  0.769701   
       90.000000 95.000000  2021       2022      0.792985  0.778881  0.779710   
       95.000000 125.000000 2021       2022      0.812907  0.824300  0.827504   
Male   0.000000  0.019178   2021       2022      0.806033  0.863578  0.799267   
       0.019178  0.076712   2021       2022      0.865388  0.886944  0.882597   
       0.076712  0.500000   2021       2022      0.763076  0.777394  0.775303   
       0.500000  1.000000   2021       2022      0.766630  0.749035  0.761216   
       1.000000  2.000000   2021       2022      0.741494  0.711596  0.743203   
       2.000000  5.000000   2021       2022      0.740349  0.751421  0.743108   
       5.000000  10.000000  2021       2022      0.624719  0.617797  0.453311   
       10.000000 15.000000  2021       2022      0.571536  0.681464  0.558209   
       15.000000 20.000000  2021       2022      0.686654  0.707973  0.742710   
       20.000000 25.000000  2021       2022      0.627483  0.599830  0.573474   
       25.000000 30.000000  2021       2022      0.583153  0.689914  0.614078   
       30.000000 35.000000  2021       2022      0.674881  0.647775  0.611479   
       35.000000 40.000000  2021       2022      0.640158  0.708754  0.674264   
       40.000000 45.000000  2021       2022      0.714766  0.666477  0.716562   
       45.000000 50.000000  2021       2022      0.687965  0.735845  0.734617   
       50.000000 55.000000  2021       2022      0.769022  0.761590  0.732257   
       55.000000 60.000000  2021       2022      0.753883  0.642267  0.704789   
       60.000000 65.000000  2021       2022      0.768321  0.704346  0.671993   
       65.000000 70.000000  2021       2022      0.665056  0.778031  0.718361   
       70.000000 75.000000  2021       2022      0.766141  0.728811  0.727694   
       75.000000 80.000000  2021       2022      0.755588  0.764418  0.737251   
       80.000000 85.000000  2021       2022      0.788737  0.780224  0.740216   
       85.000000 90.000000  2021       2022      0.812829  0.801091  0.803316   
 

In [49]:
iron_responsive_proportion.columns.name = "draw"
iron_responsive_proportion = iron_responsive_proportion[DRAWS].stack()
iron_responsive_proportion

sex     age_start  age_end     year_start  year_end  draw    
Female  0.0        0.019178    2021        2022      draw_0      0.847689
                                                     draw_1      0.843261
                                                     draw_2      0.811992
                                                     draw_3      0.889733
                                                     draw_4      0.832298
                                                                   ...   
Male    95.0       125.000000  2021        2022      draw_495    0.824822
                                                     draw_496    0.810967
                                                     draw_497    0.761300
                                                     draw_498    0.809667
                                                     draw_499    0.818629
Length: 25000, dtype: float64

In [50]:
assert (iron_responsive_proportion <= 1).all()

In [51]:
iron_responsive_proportion.sort_values()

sex     age_start  age_end    year_start  year_end  draw    
Male    10.0       15.000000  2021        2022      draw_352    0.392741
                                                    draw_74     0.411238
                                                    draw_123    0.412677
                                                    draw_25     0.416654
                                                    draw_343    0.431517
                                                                  ...   
        0.0        0.019178   2021        2022      draw_335    0.907555
                                                    draw_243    0.908144
                                                    draw_160    0.909758
Female  0.0        0.019178   2021        2022      draw_459    0.913692
                                                    draw_256    0.917918
Length: 25000, dtype: float64

## Apply fortification effect

In [52]:
hgb_mean_iron_responsive_with_fort = hgb_mean.add(hemoglobin_effect_among_responsive)
hgb_mean_iron_responsive_with_fort

sex     age_start  age_end     wealth_quintile  draw    
Female  0.0        0.019178    lowest           draw_0      138.025680
                                                draw_1      137.546410
                                                draw_2      135.434803
                                                draw_3      141.397705
                                                draw_4      146.059699
                                                               ...    
Male    95.0       125.000000  highest          draw_495    103.547427
                                                draw_496    112.915944
                                                draw_497    112.066735
                                                draw_498    105.866035
                                                draw_499    114.141086
Length: 125000, dtype: float64

In [53]:
thresholds = reshape_to_vivarium_format(
    pd.read_csv('/share/mnch/anemia/code/reference/model/anemia_thresholds.csv'),
    location.title(),
).droplevel(["age_group_name", "grp"]).reset_index()
thresholds

,sex,age_start,age_end,hgb_lower_anemic,hgb_lower_mild,hgb_lower_moderate,hgb_lower_severe,hgb_upper_anemic,hgb_upper_mild,hgb_upper_moderate,hgb_upper_severe,pregnant
0,Female,0.000000,0.019178,0,145,100,0,160,160,145,100,0
1,Female,0.019178,0.076712,0,120,85,0,135,135,120,85,0
2,Female,0.076712,0.500000,0,100,70,0,110,110,100,70,0
3,Female,0.500000,1.000000,0,100,70,0,110,110,100,70,0
4,Female,1.000000,2.000000,0,100,70,0,110,110,100,70,0
5,Female,2.000000,5.000000,0,100,70,0,110,110,100,70,0
6,Female,5.000000,10.000000,0,110,80,0,115,115,110,80,0
7,Female,10.000000,15.000000,0,100,70,0,110,110,100,70,1
8,Female,10.000000,15.000000,0,110,80,0,115,115,110,80,0
9,Female,15.000000,20.000000,0,110,80,0,120,120,110,80,0


In [54]:
assert (
    (thresholds.hgb_upper_mild == thresholds.hgb_upper_anemic).all() &
    (thresholds.hgb_lower_severe == thresholds.hgb_lower_anemic).all()
)
thresholds = thresholds.drop(columns=["hgb_upper_anemic", "hgb_lower_anemic"])

In [55]:
assert (
    (thresholds.hgb_lower_mild == thresholds.hgb_upper_moderate).all() &
    (thresholds.hgb_lower_moderate == thresholds.hgb_upper_severe).all()
)
thresholds = thresholds.drop(columns=["hgb_lower_mild", "hgb_lower_moderate"])

In [56]:
thresholds = thresholds.set_index(["sex", "age_start", "age_end", "pregnant"])
thresholds

hgb_lower_severe  hgb_upper_mild  \
sex    age_start age_end    pregnant                                     
Female 0.000000  0.019178   0                        0             160   
       0.019178  0.076712   0                        0             135   
       0.076712  0.500000   0                        0             110   
       0.500000  1.000000   0                        0             110   
       1.000000  2.000000   0                        0             110   
       2.000000  5.000000   0                        0             110   
       5.000000  10.000000  0                        0             115   
       10.000000 15.000000  1                        0             110   
                            0                        0             115   
       15.000000 20.000000  0                        0             120   
                            1                        0             110   
       20.000000 25.000000  0                        0             120   
                            1                        0             110   
       25.000000 30.000000  0                        0             120   
                            1                        0             110   
       30.000000 35.000000  0                        0             120   
                            1                        0             110   
       35.000000 40.000000  0                        0             120   
                            1                        0             110   
       40.000000 45.000000  0                        0             120   
                            1                        0             110   
       45.000000 50.000000  0                        0             120   
                            1                        0             110   
       50.000000 55.000000  0                        0             120   
                            1                        0             110   
       55.000000 60.000000  0                        0             120   
       60.000000 65.000000  0                        0             120   
       65.000000 70.000000  0                        0             120   
       70.000000 75.000000  0                        0             120   
       75.000000 80.000000  0                        0             120   
       80.000000 85.000000  0                        0             120   
       85.000000 90.000000  0                        0             120   
       90.000000 95.000000  0                        0             120   
       95.000000 125.000000 0                        0             120   
Male   0.000000  0.019178   0                        0             160   
       0.019178  0.076712   0                        0             135   
       0.076712  0.500000   0                        0             110   
       0.500000  1.000000   0                        0             110   
       1.000000  2.000000   0                        0             110   
       2.000000  5.000000   0                        0             110   
       5.000000  10.000000  0                        0             115   
       10.000000 15.000000  0                        0             115   
       15.000000 20.000000  0                        0             130   
       20.000000 25.000000  0                        0             130   
       25.000000 30.000000  0                        0             130   
       30.000000 35.000000  0                        0             130   
       35.000000 40.000000  0                        0             130   
       40.000000 45.000000  0                        0             130   
       45.000000 50.000000  0                        0             130   
       50.000000 55.000000  0                        0             130   
       55.000000 60.000000  0                        0             130   
       60.000000 65.000000  0                        0             130   
       65.000000 70.000000  0             

In [57]:
def calculate_anemia_from_mean_sd_hemoglobin(mean, sd):
    orig_index = mean.index
    result = mean.reset_index().merge(sd.reset_index(), how="outer", validate="m:1").assign(pregnant=0).merge(thresholds.reset_index(), how="left", validate="m:1")

    cdf = hemoglobin_cdf_from_mean_sd(result["mean"], result.sd)

    result["severe"] = cdf(result.hgb_upper_severe.copy()) - cdf(result.hgb_lower_severe.copy())
    result["moderate"] = cdf(result.hgb_upper_moderate.copy()) - result["severe"].copy()
    result["mild"] = cdf(result.hgb_upper_mild.copy()) - result["moderate"].copy() - result["severe"].copy()
    result["anemic"] = result["mild"] + result["moderate"] + result["severe"]
    
    return result.set_index(orig_index.names)[["severe", "moderate", "mild", "anemic"]]

In [58]:
baseline_anemia = calculate_anemia_from_mean_sd_hemoglobin(hgb_mean, hgb_sd)
baseline_anemia

severe  moderate  \
sex    age_start age_end    wealth_quintile draw                           
Female 0.0       0.019178   lowest          draw_0    0.112862  0.522239   
                                            draw_1    0.099261  0.557548   
                                            draw_2    0.048532  0.734690   
                                            draw_3    0.042398  0.586287   
                                            draw_4    0.021629  0.516715   
...                                                        ...       ...   
Male   95.0      125.000000 highest         draw_495  0.259055  0.349079   
                                            draw_496  0.063503  0.435526   
                                            draw_497  0.101301  0.410033   
                                            draw_498  0.245769  0.329388   
                                            draw_499  0.044169  0.427180   

                                                          mild    anemic  
sex    age_start age_end    wealth_quintile draw                          
Female 0.0       0.019178   lowest          draw_0    0.199909  0.835009  
                                            draw_1    0.208117  0.864926  
                                            draw_2    0.185452  0.968674  
                                            draw_3    0.261777  0.890462  
                                            draw_4    0.313810  0.852154  
...                                                        ...       ...  
Male   95.0      125.000000 highest         draw_495  0.224492  0.832625  
                                            draw_496  0.402387  0.901416  
                                            draw_497  0.348962  0.860296  
                                            draw_498  0.222421  0.797578  
                                            draw_499  0.439720  0.911068  

[125000 rows x 4 columns]

In [59]:
iron_responsive_with_fort_anemia = calculate_anemia_from_mean_sd_hemoglobin(hgb_mean_iron_responsive_with_fort.rename("mean"), hgb_sd)
iron_responsive_with_fort_anemia

severe  moderate  \
sex    age_start age_end    wealth_quintile draw                           
Female 0.0       0.019178   lowest          draw_0    0.088801  0.481584   
                                            draw_1    0.076181  0.510330   
                                            draw_2    0.032785  0.655381   
                                            draw_3    0.030068  0.508247   
                                            draw_4    0.015202  0.426505   
...                                                        ...       ...   
Male   95.0      125.000000 highest         draw_495  0.220053  0.333613   
                                            draw_496  0.043114  0.360556   
                                            draw_497  0.073958  0.356925   
                                            draw_498  0.209582  0.313764   
                                            draw_499  0.028977  0.340282   

                                                          mild    anemic  
sex    age_start age_end    wealth_quintile draw                          
Female 0.0       0.019178   lowest          draw_0    0.212520  0.782905  
                                            draw_1    0.226505  0.813016  
                                            draw_2    0.252222  0.940388  
                                            draw_3    0.290791  0.829107  
                                            draw_4    0.330996  0.772703  
...                                                        ...       ...  
Male   95.0      125.000000 highest         draw_495  0.235669  0.789335  
                                            draw_496  0.432545  0.836215  
                                            draw_497  0.365999  0.796882  
                                            draw_498  0.229536  0.752882  
                                            draw_499  0.472887  0.842147  

[125000 rows x 4 columns]

In [60]:
affected_by_intervention = (delta_effective_coverage * iron_responsive_proportion).droplevel(["year_start", "year_end"])
affected_by_intervention

age_start  age_end     sex     wealth_quintile  draw    
0.0        0.019178    Female  fourth           draw_0      0.082534
                                                draw_1      0.082102
                                                draw_2      0.079058
                                                draw_3      0.086627
                                                draw_4      0.081035
                                                              ...   
95.0       125.000000  Male    second           draw_495    0.131164
                                                draw_496    0.128961
                                                draw_497    0.121063
                                                draw_498    0.128754
                                                draw_499    0.130180
Length: 125000, dtype: float64

In [61]:
# NOTE: I am pretty sure this is correct, but it is quite difficult to think through *why*.

# First, observe that people in the population who start as non-anemic never factor into
# any of these metrics. If they started non-anemic, our counterfactual can only shift them up,
# so they did not change anemia categories between scenarios and hence have no importance to
# anemia prevalence or YLDs.

# So you can think of our hemoglobin distributions as only being of interest in the part
# of them below the anemia threshold.

# *Within* this subpopulation, we make the assumption that iron-responsive and non-iron-responsive
# anemic people have the same distributions of hemoglobin. This is probably not true, but GBD doesn't
# give us anything better.

# So you can think of our original distribution as a mixture of two parts, which are the same.
# Then we shift one of those parts (the iron-responsive part that received new effective coverage) and calculate all these stats from
# that new distribution.

# Our *actual* result should be about a mixture distribution that has the non-iron-responsive part
# the same as in baseline, with the shifted iron-responsive part.
# For all these metrics, it is pretty straightforward to see that the metric in such a mixture is
# just a weighted average of the metrics in each part, since they all depend on CDFs which combine
# this way.

intervention_anemia = baseline_anemia.mul(1 - affected_by_intervention, axis=0) + iron_responsive_with_fort_anemia.mul(affected_by_intervention, axis=0) 
intervention_anemia

severe  moderate  \
sex    age_start age_end    wealth_quintile draw                           
Female 0.0       0.019178   fourth          draw_0    0.044874  0.451407   
                                            draw_1    0.036425  0.471258   
                                            draw_10   0.003422  0.474946   
                                            draw_100  0.021962  0.380662   
                                            draw_101  0.066293  0.460752   
...                                                        ...       ...   
Male   95.0      125.000000 second          draw_95   0.052438  0.473695   
                                            draw_96   0.219572  0.417379   
                                            draw_97   0.003032  0.342365   
                                            draw_98   0.234179  0.345322   
                                            draw_99   0.148135  0.470176   

                                                          mild    anemic  
sex    age_start age_end    wealth_quintile draw                          
Female 0.0       0.019178   fourth          draw_0    0.252924  0.749205  
                                            draw_1    0.272449  0.780132  
                                            draw_10   0.438699  0.917067  
                                            draw_100  0.283336  0.685960  
                                            draw_101  0.227987  0.755032  
...                                                        ...       ...  
Male   95.0      125.000000 second          draw_95   0.411835  0.937968  
                                            draw_96   0.255955  0.892906  
                                            draw_97   0.639778  0.985175  
                                            draw_98   0.234232  0.813732  
                                            draw_99   0.302544  0.920856  

[125000 rows x 4 columns]

In [62]:
(baseline_anemia - intervention_anemia).sort_values("anemic")

severe  moderate  \
sex    age_start age_end   wealth_quintile draw                           
Male   0.0       0.019178  lowest          draw_363  0.000012  0.018021   
                                           draw_353  0.000026  0.012698   
                                           draw_161  0.000066  0.011672   
                                           draw_81   0.000025  0.020342   
                                           draw_174  0.000211  0.005902   
...                                                       ...       ...   
       10.0      15.000000 lowest          draw_96   0.000010  0.021722   
Female 10.0      15.000000 lowest          draw_31   0.000059  0.033108   
       90.0      95.000000 lowest          draw_295  0.000035  0.020360   
       50.0      55.000000 lowest          draw_267  0.000062  0.020651   
       75.0      80.000000 lowest          draw_199  0.000039  0.019191   

                                                         mild    anemic  
sex    age_start age_end   wealth_quintile draw                          
Male   0.0       0.019178  lowest          draw_363 -0.018023  0.000009  
                                           draw_353 -0.012712  0.000012  
                                           draw_161 -0.011710  0.000027  
                                           draw_81  -0.020337  0.000030  
                                           draw_174 -0.006079  0.000034  
...                                                       ...       ...  
       10.0      15.000000 lowest          draw_96   0.018190  0.039922  
Female 10.0      15.000000 lowest          draw_31   0.007130  0.040297  
       90.0      95.000000 lowest          draw_295  0.019908  0.040303  
       50.0      55.000000 lowest          draw_267  0.020049  0.040763  
       75.0      80.000000 lowest          draw_199  0.022303  0.041532  

[125000 rows x 4 columns]

In [63]:
def anemia_to_yld_rates(anemia):
    disability_weights = pd.read_hdf('/mnt/team/simulation_science/costeffectiveness/auxiliary_data/GBD_2021/02_processed_data/disability_weight/sequela/all/all.hdf')
    disability_weights = disability_weights[disability_weights.healthstate.isin(['anemia_mild', 'anemia_mod', 'anemia_sev'])].set_index('healthstate').filter(like='draw_')
    disability_weights.columns.name = 'draw'
    disability_weights = disability_weights.stack().rename('disability_weight').reset_index()
    display(disability_weights)

    orig_index = anemia.index
    anemia = anemia.reset_index().merge(
        disability_weights[disability_weights.healthstate == 'anemia_mild'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'mild_dw'}),
        validate="m:1",
    ).merge(
        disability_weights[disability_weights.healthstate == 'anemia_mod'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'moderate_dw'}),
        validate="m:1",
    ).merge(
        disability_weights[disability_weights.healthstate == 'anemia_sev'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'severe_dw'}),
        validate="m:1",
    )

    anemia["mild_yld_rate"] = anemia.mild * anemia.mild_dw
    anemia["moderate_yld_rate"] = anemia.moderate * anemia.moderate_dw
    anemia["severe_yld_rate"] = anemia.severe * anemia.severe_dw
    anemia['anemic_yld_rate'] = anemia['mild_yld_rate'] + anemia['moderate_yld_rate'] + anemia['severe_yld_rate']

    return anemia.set_index(orig_index.names).filter(like='yld_rate')

In [64]:
baseline_anemia_yld_rates = anemia_to_yld_rates(baseline_anemia)
baseline_anemia_yld_rates

,healthstate,draw,disability_weight
0,anemia_mild,draw_0,0.002420
1,anemia_mild,draw_1,0.003172
2,anemia_mild,draw_2,0.002644
3,anemia_mild,draw_3,0.003085
4,anemia_mild,draw_4,0.001845
...,...,...,...
2995,anemia_sev,draw_995,0.174969
2996,anemia_sev,draw_996,0.086798
2997,anemia_sev,draw_997,0.131996
2998,anemia_sev,draw_998,0.124427


mild_yld_rate  \
sex    age_start age_end    wealth_quintile draw                      
Female 0.0       0.019178   lowest          draw_0         0.000484   
                            second          draw_0         0.000528   
                            middle          draw_0         0.000570   
                            fourth          draw_0         0.000612   
                            highest         draw_0         0.000664   
...                                                             ...   
Male   95.0      125.000000 lowest          draw_499       0.001134   
                            second          draw_499       0.001229   
                            middle          draw_499       0.001249   
                            fourth          draw_499       0.001281   
                            highest         draw_499       0.001390   

                                                      moderate_yld_rate  \
sex    age_start age_end    wealth_quintile draw                          
Female 0.0       0.019178   lowest          draw_0             0.030969   
                            second          draw_0             0.029801   
                            middle          draw_0             0.027580   
                            fourth          draw_0             0.027066   
                            highest         draw_0             0.022656   
...                                                                 ...   
Male   95.0      125.000000 lowest          draw_499           0.021306   
                            second          draw_499           0.020147   
                            middle          draw_499           0.019567   
                            fourth          draw_499           0.019403   
                            highest         draw_499           0.017996   

                                                      severe_yld_rate  \
sex    age_start age_end    wealth_quintile draw                        
Female 0.0       0.019178   lowest          draw_0           0.022453   
                            second          draw_0           0.016943   
                            middle          draw_0           0.012010   
                            fourth          draw_0           0.009117   
                            highest         draw_0           0.004930   
...                                                               ...   
Male   95.0      125.000000 lowest          draw_499         0.008764   
                            second          draw_499         0.007178   
                            middle          draw_499         0.006991   
                            fourth          draw_499         0.006364   
                            highest         draw_499         0.004714   

                                                      anemic_yld_rate  
sex    age_start age_end    wealth_quintile draw                       
Female 0.0       0.019178   lowest          draw_0           0.053906  
                            second          draw_0           0.047272  
                            middle          draw_0           0.040160  
                            fourth          draw_0           0.036795  
                            highest         draw_0           0.028250  
...                                                               ...  
Male   95.0      125.000000 lowest          draw_499         0.031204  
                            second          draw_499         0.028554  
                            middle          draw_499         0.027807  
                            fourth          draw_499         0.027048  
                            highest         draw_499         0.024100  

[125000 rows x 4 columns]

In [65]:
intervention_anemia_yld_rates = anemia_to_yld_rates(intervention_anemia)
intervention_anemia_yld_rates

,healthstate,draw,disability_weight
0,anemia_mild,draw_0,0.002420
1,anemia_mild,draw_1,0.003172
2,anemia_mild,draw_2,0.002644
3,anemia_mild,draw_3,0.003085
4,anemia_mild,draw_4,0.001845
...,...,...,...
2995,anemia_sev,draw_995,0.174969
2996,anemia_sev,draw_996,0.086798
2997,anemia_sev,draw_997,0.131996
2998,anemia_sev,draw_998,0.124427


mild_yld_rate  \
sex    age_start age_end    wealth_quintile draw                     
Female 0.0       0.019178   fourth          draw_0        0.000612   
                            highest         draw_0        0.000662   
                            lowest          draw_0        0.000489   
                            middle          draw_0        0.000571   
                            second          draw_0        0.000531   
...                                                            ...   
Male   95.0      125.000000 fourth          draw_99       0.001311   
                            highest         draw_99       0.001413   
                            lowest          draw_99       0.001194   
                            middle          draw_99       0.001285   
                            second          draw_99       0.001271   

                                                     moderate_yld_rate  \
sex    age_start age_end    wealth_quintile draw                         
Female 0.0       0.019178   fourth          draw_0            0.026769   
                            highest         draw_0            0.022428   
                            lowest          draw_0            0.030555   
                            middle          draw_0            0.027260   
                            second          draw_0            0.029417   
...                                                                ...   
Male   95.0      125.000000 fourth          draw_99           0.019400   
                            highest         draw_99           0.019071   
                            lowest          draw_99           0.019946   
                            middle          draw_99           0.019249   
                            second          draw_99           0.019597   

                                                     severe_yld_rate  \
sex    age_start age_end    wealth_quintile draw                       
Female 0.0       0.019178   fourth          draw_0          0.008927   
                            highest         draw_0          0.004856   
                            lowest          draw_0          0.021630   
                            middle          draw_0          0.011724   
                            second          draw_0          0.016422   
...                                                              ...   
Male   95.0      125.000000 fourth          draw_99         0.016454   
                            highest         draw_99         0.013564   
                            lowest          draw_99         0.019881   
                            middle          draw_99         0.017386   
                            second          draw_99         0.017596   

                                                     anemic_yld_rate  
sex    age_start age_end    wealth_quintile draw                      
Female 0.0       0.019178   fourth          draw_0          0.036308  
                            highest         draw_0          0.027946  
                            lowest          draw_0          0.052674  
                            middle          draw_0          0.039555  
                            second          draw_0          0.046370  
...                                                              ...  
Male   95.0      125.000000 fourth          draw_99         0.037165  
                            highest         draw_99         0.034049  
                            lowest          draw_99         0.041021  
                            middle          draw_99         0.037920  
                            second          draw_99         0.038464  

[125000 rows x 4 columns]

In [66]:
(baseline_anemia_yld_rates - intervention_anemia_yld_rates).sort_values("anemic_yld_rate")

mild_yld_rate  \
sex    age_start age_end   wealth_quintile draw                      
Male   25.0      30.000000 highest         draw_38    6.229362e-07   
                                           draw_472   2.757823e-06   
                                           draw_420   2.208403e-06   
                                           draw_34    1.614481e-06   
                                           draw_106   1.468362e-06   
...                                                            ...   
Female 55.0      60.000000 lowest          draw_356  -4.823421e-05   
       90.0      95.000000 lowest          draw_187  -1.529490e-04   
Male   0.0       0.019178  lowest          draw_356  -1.257577e-04   
                                           draw_53   -4.065726e-04   
                                           draw_408  -1.882839e-04   

                                                     moderate_yld_rate  \
sex    age_start age_end   wealth_quintile draw                          
Male   25.0      30.000000 highest         draw_38            0.000002   
                                           draw_472           0.000001   
                                           draw_420           0.000002   
                                           draw_34            0.000002   
                                           draw_106           0.000003   
...                                                                ...   
Female 55.0      60.000000 lowest          draw_356           0.002731   
       90.0      95.000000 lowest          draw_187           0.002913   
Male   0.0       0.019178  lowest          draw_356           0.002924   
                                           draw_53            0.003565   
                                           draw_408           0.003366   

                                                     severe_yld_rate  \
sex    age_start age_end   wealth_quintile draw                        
Male   25.0      30.000000 highest         draw_38      2.533508e-07   
                                           draw_472     3.347145e-08   
                                           draw_420     1.153488e-07   
                                           draw_34      2.739672e-07   
                                           draw_106     3.220342e-07   
...                                                              ...   
Female 55.0      60.000000 lowest          draw_356     7.537774e-05   
       90.0      95.000000 lowest          draw_187     3.218253e-05   
Male   0.0       0.019178  lowest          draw_356     1.455219e-05   
                                           draw_53      2.686062e-06   
                                           draw_408     2.322762e-06   

                                                     anemic_yld_rate  
sex    age_start age_end   wealth_quintile draw                       
Male   25.0      30.000000 highest         draw_38          0.000003  
                                           draw_472         0.000004  
                                           draw_420         0.000004  
                                           draw_34          0.000004  
                                           draw_106         0.000005  
...                                                              ...  
Female 55.0      60.000000 lowest          draw_356         0.002759  
       90.0      95.000000 lowest          draw_187         0.002792  
Male   0.0       0.019178  lowest          draw_356         0.002813  
                                           draw_53          0.003161  
                                           draw_408         0.003180  

[125000 rows x 4 columns]

In [67]:
assert ((baseline_anemia_yld_rates - intervention_anemia_yld_rates).anemic_yld_rate > 0).all()

In [68]:
baseline_ylds = (baseline_anemia_yld_rates.anemic_yld_rate.unstack("draw").mean(axis=1) * non_pregnant_pop)
baseline_ylds

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    fourth             424.729902
                               highest            291.089988
                               lowest             736.111714
                               middle             521.550200
                               second             650.757670
                                                     ...    
Male    95.0       125.000000  fourth              76.888359
                               highest             74.821911
                               lowest              89.386189
                               middle              73.428445
                               second              72.903842
Length: 250, dtype: float64

In [69]:
intervention_ylds = (intervention_anemia_yld_rates.anemic_yld_rate.unstack("draw").mean(axis=1) * non_pregnant_pop)
intervention_ylds

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    fourth             417.794541
                               highest            286.940313
                               lowest             718.816995
                               middle             512.164362
                               second             637.093867
                                                     ...    
Male    95.0       125.000000  fourth              75.907210
                               highest             74.104169
                               lowest              87.128066
                               middle              72.321294
                               second              71.411351
Length: 250, dtype: float64

In [70]:
baseline_ylds.groupby(["wealth_quintile"]).sum() - intervention_ylds.groupby(["wealth_quintile"]).sum()

wealth_quintile
fourth     12886.589644
highest     7340.157025
lowest     34183.951198
middle     16172.065364
second     24198.977873
dtype: float64

In [71]:
ylds = pd.concat([
    baseline_ylds.rename("value").reset_index().assign(scenario="baseline"),
    intervention_ylds.rename("value").reset_index().assign(scenario="intervention")
], ignore_index=True)
ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,fourth,424.729902,baseline
1,Female,0.0,0.019178,highest,291.089988,baseline
2,Female,0.0,0.019178,lowest,736.111714,baseline
3,Female,0.0,0.019178,middle,521.550200,baseline
4,Female,0.0,0.019178,second,650.757670,baseline
...,...,...,...,...,...,...
495,Male,95.0,125.000000,fourth,75.907210,intervention
496,Male,95.0,125.000000,highest,74.104169,intervention
497,Male,95.0,125.000000,lowest,87.128066,intervention
498,Male,95.0,125.000000,middle,72.321294,intervention


In [72]:
results_dir = f'./results/{vehicle.lower()}/{location.lower()}/{intervention_scenario.lower()}'

In [73]:
path = f'{results_dir}/ylds.parquet'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ylds.to_parquet(path)

In [74]:
baseline_anemia_prevalence = baseline_anemia['anemic'].unstack("draw").mean(axis=1)
baseline_anemia_prevalence

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    fourth             0.828338
                               highest            0.748522
                               lowest             0.900708
                               middle             0.835819
                               second             0.876689
                                                    ...   
Male    95.0       125.000000  fourth             0.891835
                               highest            0.880907
                               lowest             0.910303
                               middle             0.890964
                               second             0.899321
Length: 250, dtype: float64

In [75]:
baseline_anemia_cases = baseline_anemia_prevalence.mul(non_pregnant_pop, axis=0)
baseline_anemia_cases

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    fourth             11611.609394
                               highest             9753.105130
                               lowest             15276.458404
                               middle             13328.773023
                               second             14808.048479
                                                      ...     
Male    95.0       125.000000  fourth              1586.629929
                               highest             1671.057535
                               lowest              1681.299915
                               middle              1480.290163
                               second              1453.600055
Length: 250, dtype: float64

In [76]:
baseline_anemia_cases.sum() / non_pregnant_pop.sum()

0.45867407015636424

In [77]:
baseline_anemia_cases.groupby(["wealth_quintile"]).sum() / non_pregnant_pop.groupby(["wealth_quintile"]).sum()

wealth_quintile
fourth     0.434832
highest    0.350655
lowest     0.556781
middle     0.454171
second     0.498840
dtype: float64

In [78]:
intervention_anemia_prevalence = intervention_anemia['anemic'].unstack("draw").mean(axis=1)
intervention_anemia_prevalence

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    fourth             0.822236
                               highest            0.742668
                               lowest             0.893422
                               middle             0.829128
                               second             0.869586
                                                    ...   
Male    95.0       125.000000  fourth             0.887867
                               highest            0.877611
                               lowest             0.903731
                               middle             0.886299
                               second             0.893363
Length: 250, dtype: float64

In [79]:
anemia_prevalence = pd.concat([
    baseline_anemia_prevalence.rename("value").reset_index().assign(scenario="baseline"),
    intervention_anemia_prevalence.rename("value").reset_index().assign(scenario="intervention")
], ignore_index=True)
anemia_prevalence

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,fourth,0.828338,baseline
1,Female,0.0,0.019178,highest,0.748522,baseline
2,Female,0.0,0.019178,lowest,0.900708,baseline
3,Female,0.0,0.019178,middle,0.835819,baseline
4,Female,0.0,0.019178,second,0.876689,baseline
...,...,...,...,...,...,...
495,Male,95.0,125.000000,fourth,0.887867,intervention
496,Male,95.0,125.000000,highest,0.877611,intervention
497,Male,95.0,125.000000,lowest,0.903731,intervention
498,Male,95.0,125.000000,middle,0.886299,intervention


In [80]:
path = f'{results_dir}/anemia_prevalence.parquet'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
anemia_prevalence.to_parquet(path)

In [81]:
intervention_anemia_cases = intervention_anemia_prevalence.mul(non_pregnant_pop, axis=0)
intervention_anemia_cases

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    fourth             11526.066957
                               highest             9676.823401
                               lowest             15152.887300
                               middle             13222.071153
                               second             14688.083843
                                                      ...     
Male    95.0       125.000000  fourth              1579.569992
                               highest             1664.804547
                               lowest              1669.162311
                               middle              1472.539174
                               second              1443.969656
Length: 250, dtype: float64

In [82]:
intervention_anemia_cases.sum() / non_pregnant_pop.sum()

0.44907246654912514

In [83]:
intervention_anemia_cases.groupby(["wealth_quintile"]).sum() / non_pregnant_pop.groupby(["wealth_quintile"]).sum()

wealth_quintile
fourth     0.427592
highest    0.345809
lowest     0.541407
middle     0.445505
second     0.486845
dtype: float64

In [84]:
(baseline_anemia_cases.groupby(["wealth_quintile"]).sum() - intervention_anemia_cases.groupby(["wealth_quintile"]).sum()).map(lambda x: f'{round(x):,.0f}')

wealth_quintile
fourth     326,909
highest    217,792
lowest     680,758
middle     387,556
second     533,975
dtype: object

In [85]:
anemia_cases = pd.concat([
    baseline_anemia_cases.rename("value").reset_index().assign(scenario="baseline"),
    intervention_anemia_cases.rename("value").reset_index().assign(scenario="intervention")
], ignore_index=True)
anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,fourth,11611.609394,baseline
1,Female,0.0,0.019178,highest,9753.105130,baseline
2,Female,0.0,0.019178,lowest,15276.458404,baseline
3,Female,0.0,0.019178,middle,13328.773023,baseline
4,Female,0.0,0.019178,second,14808.048479,baseline
...,...,...,...,...,...,...
495,Male,95.0,125.000000,fourth,1579.569992,intervention
496,Male,95.0,125.000000,highest,1664.804547,intervention
497,Male,95.0,125.000000,lowest,1669.162311,intervention
498,Male,95.0,125.000000,middle,1472.539174,intervention


In [86]:
path = f'{results_dir}/anemia_cases.parquet'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
anemia_cases.to_parquet(path)